In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 15.4 MB/s eta 0:00:00:00:0100:01


In [3]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np

In [4]:
base_path = Path("/content/drive/MyDrive/Colab Notebooks/U-net/MSLesSeg Dataset/")

dataset_dict = {}

In [5]:
def get_patient_number(p_id):
    num_str = ''.join(filter(str.isdigit, p_id))
    return int(num_str) if num_str else 0

for patient_dir in base_path.iterdir():
    if not patient_dir.is_dir():
        continue

    patient_id = patient_dir.name
    p_num = get_patient_number(patient_id)
    dataset_dict[patient_id] = {}

    if p_num <= 53:
        for timeline_dir in patient_dir.iterdir():
            if not timeline_dir.is_dir():
                continue

            timeline_id = timeline_dir.name
            dataset_dict[patient_id][timeline_id] = {}

            for file_path in timeline_dir.glob("*.nii.gz"):
                filename = file_path.name.replace('.nii.gz', '')
                modality = filename.split('_')[-1]
                dataset_dict[patient_id][timeline_id][modality] = file_path

    else:
        for file_path in patient_dir.glob("*.nii.gz"):
            filename = file_path.name.replace('.nii.gz', '')
            modality = filename.split('_')[-1]
            dataset_dict[patient_id][modality] = file_path

In [6]:
sample_file_path = str(dataset_dict['P28']['T1']['T2'])
sitk_image = sitk.ReadImage(sample_file_path)

image_array = sitk.GetArrayFromImage(sitk_image)

print(f"Размерность массива: {image_array.shape}")

Размерность массива: (182, 218, 182)


In [7]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

In [8]:
def explore_3D_array(arr: np.ndarray, cmap: str = 'grey'):

  def fn(SLICE):
    plt.figure(figsize=(7, 7))
    plt.imshow(arr[SLICE, :, :], cmap=cmap)
    plt.show()

  interact(fn, SLICE=(0, arr.shape[0]-1))


explore_3D_array(image_array)

interactive(children=(IntSlider(value=90, description='SLICE', max=181), Output()), _dom_classes=('widget-inte…

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
all_patients = list(dataset_dict.keys())

In [11]:
test_patients = [p for p in all_patients if get_patient_number(p) >= 54]
remaining_patients = [p for p in all_patients if get_patient_number(p) < 54]

In [12]:
train_patients, val_patients = train_test_split(
    remaining_patients,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(len(all_patients))
print(f"Train: {len(train_patients)} | Val: {len(val_patients)} | Test: {len(test_patients)}")

75
Train: 42 | Val: 11 | Test: 22


In [13]:
train_dict = {p: dataset_dict[p] for p in train_patients}
val_dict = {p: dataset_dict[p] for p in val_patients}
test_dict = {p: dataset_dict[p] for p in test_patients}

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.io import read_image
import torch.optim as optim
import gc

In [16]:
def center_crop(layer, target_layer):
    _, _, d, h, w = layer.size()
    _, _, td, th, tw = target_layer.size()

    dd = (d - td) // 2
    dh = (h - th) // 2
    dw = (w - tw) // 2

    return layer[:, :, dd:dd+td, dh:dh+th, dw:dw+tw]

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.pool1 = nn.MaxPool3d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool3d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool3d(2)
        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool3d(2)
        self.bottom = DoubleConv(512, 1024)

        self.conv_bn=nn.BatchNorm3d(1024)
        # Decoder
        self.up4  = nn.ConvTranspose3d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3  = nn.ConvTranspose3d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2  = nn.ConvTranspose3d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1  = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.out_conv = nn.Conv3d(64, out_ch, kernel_size=1)

    def forward(self, x):
        #Encoder
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x4 = self.down4(self.pool3(x3))
        x5 = self.bottom(self.pool4(x4))

        #Decoder
        x = self.up4(x5)
        x4_cropped = center_crop(x4, x)
        x = torch.cat([x4_cropped, x], dim=1)
        x = self.dec4(x)

        x = self.up3(x)
        x3_cropped = center_crop(x3, x)
        x = torch.cat([x3_cropped, x], dim=1)
        x = self.dec3(x)

        x = self.up2(x)
        x2_cropped = center_crop(x2, x)
        x = torch.cat([x2_cropped, x], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x1_cropped = center_crop(x1, x)
        x = torch.cat([x1_cropped, x], dim=1)
        x = self.dec1(x)

        logits = self.out_conv(x)
        return logits

In [17]:
def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    probs = probs.contiguous().view(probs.size(0), -1)    #(1, H, W) -> (1,N): N = H*W
    targets = targets.contiguous().view(targets.size(0), -1)  #(1, H, W) -> (1,N): N = H*W

    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()

In [18]:
def flatten_dataset(data_dict):
    flat_samples = []
    for p_id, p_data in data_dict.items():
        has_timelines = any(isinstance(v, dict) for v in p_data.values())

        if has_timelines:
            for timeline_id, modalities in p_data.items():
                if isinstance(modalities, dict):
                    sample = {'patient_id': p_id, 'timeline': timeline_id}
                    sample.update(modalities)
                    flat_samples.append(sample)
        else:
            sample = {'patient_id': p_id, 'timeline': 'baseline'}
            sample.update(p_data)
            flat_samples.append(sample)

    return flat_samples


train_samples = flatten_dataset(train_dict)
val_samples = flatten_dataset(val_dict)
test_samples = flatten_dataset(test_dict)

In [19]:
class MSLesSegDataset(Dataset):
    def __init__(self, samples_list, modalities=['T1', 'T2', 'FLAIR']):
        self.samples = samples_list
        self.modalities = modalities

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        images = []
        for mod in self.modalities:
            file_path = str(sample[mod])
            sitk_img = sitk.ReadImage(file_path)
            img_array = sitk.GetArrayFromImage(sitk_img)

            img_array = (img_array - np.mean(img_array)) / (np.std(img_array) + 1e-8)
            images.append(img_array)

        image_tensor = np.stack(images, axis=0)
        image_tensor = torch.tensor(image_tensor, dtype=torch.float32)

        mask_path = str(sample['MASK'])
        sitk_mask = sitk.ReadImage(mask_path)
        mask_array = sitk.GetArrayFromImage(sitk_mask)
        mask_tensor = torch.tensor(mask_array, dtype=torch.long).unsqueeze(0)

        return image_tensor, mask_tensor

train_dataset = MSLesSegDataset(train_samples)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [20]:
batch_size = 1
modalities = ['FLAIR', 'T1', 'T2']

train_dataset = MSLesSegDataset(train_samples, modalities=modalities)
val_dataset   = MSLesSegDataset(val_samples,   modalities=modalities)
test_dataset  = MSLesSegDataset(test_samples,  modalities=modalities)

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=1)

In [21]:
images, masks = next(iter(train_loader))
print(images.shape)
print(masks.shape)

torch.Size([1, 3, 182, 218, 182])
torch.Size([1, 1, 182, 218, 182])


In [22]:
from tqdm.auto import tqdm

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
gc.collect()

model = UNet(in_ch=len(modalities), out_ch=1)
model = model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs  = 100
lambda_dice = 1.0

train_losses = []
val_losses   = []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)

        #masks_c = center_crop(masks, logits)

        loss = criterion(logits, masks.float())
        loss += dice_loss(logits, masks) * lambda_dice

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device)

            logits = model(images)
            #masks_c = center_crop(masks, logits)

            loss = criterion(logits, masks.float())
            loss += dice_loss(logits, masks) * lambda_dice

            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss = {train_loss:.4f} | val_loss = {val_loss:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 34.86 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)